In [ ]:
# Create Spark Session
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("Repartition & Joins")
    .master("local[*]")
    .getOrCreate()
)
    

In [ ]:
# Create employee dataset (emp_1)
emp_data_1=[
     ["001","101","vaibhav","21","male","1200000"],
    ["002","102","dipak","22","male","20000"],
    ["003","103","tejas","23","female","2095"],
    ["004","104","anita","24","female","35000"],
    ["005","105","rohit","25","male","45000"],
    ["006","106","sneha","26","female","50000"],
    ["007","107","rahul","27","male","60000"],
    ["008","108","priya","28","female","70000"],
    ["009","109","amit","29","male","80000"],
    ["010","110","kavita","30","female","90000"],
    ["011","111","vikas","31","male","100000"],
    ["012","112","neha","32","female","110000"]
]
emp_schema_1="employee_id string,dept_id string,name string,age string,gender string,salary string"

emp_data_2=[
    ["1","vaibhav","101","shirdi"],
    ["2","dipak","102","mumbai"],
    ["3","aryan","103","panvel"],
    ["4","tejas","101","magarpatta"],
    ["5","roshan","102","korhale"]
]
emp_schema_2="id string,name string,dept_id string,city string"

In [ ]:
# Creating DataFrame
emp_1=spark.createDataFrame(data=emp_data_1,schema=emp_schema_1)
emp_2=spark.createDataFrame(data=emp_data_2,schema=emp_schema_2)


In [4]:
emp_1.show()
emp_2.show()

+-----------+-------+-------+---+------+-------+
|employee_id|dept_id|   name|age|gender| salary|
+-----------+-------+-------+---+------+-------+
|        001|    101|vaibhav| 21|  male|1200000|
|        002|    102|  dipak| 22|  male|  20000|
|        003|    103|  tejas| 23|female|   2095|
|        004|    104|  anita| 24|female|  35000|
|        005|    105|  rohit| 25|  male|  45000|
|        006|    106|  sneha| 26|female|  50000|
|        007|    107|  rahul| 27|  male|  60000|
|        008|    108|  priya| 28|female|  70000|
|        009|    109|   amit| 29|  male|  80000|
|        010|    110| kavita| 30|female|  90000|
|        011|    111|  vikas| 31|  male| 100000|
|        012|    112|   neha| 32|female| 110000|
+-----------+-------+-------+---+------+-------+

+---+-------+-------+----------+
| id|   name|dept_id|      city|
+---+-------+-------+----------+
|  1|vaibhav|    101|    shirdi|
|  2|  dipak|    102|    mumbai|
|  3|  aryan|    103|    panvel|
|  4|  tejas|    

In [ ]:
# Repartition emp_1 based on dept_id
from pyspark.sql.functions import spark_partition_id,col
emp_1=emp_1.repartition(4,"dept_id").withColumn("partition_num",spark_partition_id())


In [12]:
emp_1.show()


+-----------+-------+-------+---+------+-------+-------------+
|employee_id|dept_id|   name|age|gender| salary|partition_num|
+-----------+-------+-------+---+------+-------+-------------+
|        002|    102|  dipak| 22|  male|  20000|            0|
|        007|    107|  rahul| 27|  male|  60000|            0|
|        010|    110| kavita| 30|female|  90000|            0|
|        011|    111|  vikas| 31|  male| 100000|            0|
|        005|    105|  rohit| 25|  male|  45000|            1|
|        009|    109|   amit| 29|  male|  80000|            1|
|        004|    104|  anita| 24|female|  35000|            2|
|        006|    106|  sneha| 26|female|  50000|            2|
|        001|    101|vaibhav| 21|  male|1200000|            3|
|        003|    103|  tejas| 23|female|   2095|            3|
|        008|    108|  priya| 28|female|  70000|            3|
|        012|    112|   neha| 32|female| 110000|            3|
+-----------+-------+-------+---+------+-------+-------

In [20]:
emp_partition=emp_2.repartition(4,"id").withColumn("partitioned",spark_partition_id())

In [21]:
emp_partition.show()

+---+-------+-------+----------+-----------+
| id|   name|dept_id|      city|partitioned|
+---+-------+-------+----------+-----------+
|  1|vaibhav|    101|    shirdi|          0|
|  2|  dipak|    102|    mumbai|          1|
|  5| roshan|    102|   korhale|          1|
|  3|  aryan|    103|    panvel|          2|
|  4|  tejas|    101|magarpatta|          2|
+---+-------+-------+----------+-----------+



In [ ]:
# Perform INNER JOIN (emp_1 & emp_2)
df_joined=emp_1.join(emp_2,how="inner",on=emp_1.dept_id == emp_2.dept_id)

In [28]:
df_joined.show()

+-----------+-------+-------+---+------+-------+-------------+---+-------+-------+----------+
|employee_id|dept_id|   name|age|gender| salary|partition_num| id|   name|dept_id|      city|
+-----------+-------+-------+---+------+-------+-------------+---+-------+-------+----------+
|        001|    101|vaibhav| 21|  male|1200000|            3|  1|vaibhav|    101|    shirdi|
|        002|    102|  dipak| 22|  male|  20000|            0|  2|  dipak|    102|    mumbai|
|        003|    103|  tejas| 23|female|   2095|            3|  3|  aryan|    103|    panvel|
|        001|    101|vaibhav| 21|  male|1200000|            3|  4|  tejas|    101|magarpatta|
|        002|    102|  dipak| 22|  male|  20000|            0|  5| roshan|    102|   korhale|
+-----------+-------+-------+---+------+-------+-------------+---+-------+-------+----------+



In [30]:
emp_joined_2=emp_2.join(emp_1,how="inner",on=emp_2.dept_id==emp_1.dept_id)

In [31]:
emp_joined_2.show()

+---+-------+-------+----------+-----------+-------+-------+---+------+-------+-------------+
| id|   name|dept_id|      city|employee_id|dept_id|   name|age|gender| salary|partition_num|
+---+-------+-------+----------+-----------+-------+-------+---+------+-------+-------------+
|  5| roshan|    102|   korhale|        002|    102|  dipak| 22|  male|  20000|            0|
|  2|  dipak|    102|    mumbai|        002|    102|  dipak| 22|  male|  20000|            0|
|  4|  tejas|    101|magarpatta|        001|    101|vaibhav| 21|  male|1200000|            3|
|  1|vaibhav|    101|    shirdi|        001|    101|vaibhav| 21|  male|1200000|            3|
|  3|  aryan|    103|    panvel|        003|    103|  tejas| 23|female|   2095|            3|
+---+-------+-------+----------+-----------+-------+-------+---+------+-------+-------------+

